# Asset Category ETL & Data Cleaning
### AMS Legacy Migration to Smart Microsoft SQL Server (`categories` table)

---

## 1. Overview & Business Context
This notebook performs data cleaning, normalization, and relational mapping on the legacy AMS (Asset Management System) categories dataset (`data/ams/tb_assets_category_202609291319.csv`).

The goal is to prepare clean, verified data ready for direct import into the Microsoft SQL Server **`categories`** table for the new **Smart** application.

### Target Microsoft SQL Server Schema (`categories`)
| Column Name | SQL Type | Constraints | Description |
|:---|:---|:---|:---|
| `id` | `BIGINT` | `IDENTITY(1,1) PRIMARY KEY` | Auto-incrementing identifier |
| `code` | `VARCHAR(4)` | `UNIQUE NOT NULL` | Standardized 2-4 uppercase letter category code |
| `name` | `NVARCHAR(255)` | `NOT NULL` | Standardized category name in Title Case |
| `created_at` | `DATETIME` | `NOT NULL` | Record creation timestamp (`YYYY-MM-DD HH:MM:SS`) |
| `updated_at` | `DATETIME` | `NOT NULL` | Record update timestamp (`YYYY-MM-DD HH:MM:SS`) |

---

## 2. Key Data Quality Findings & ETL Decisions
During data inspection of the 128 legacy rows in `tb_assets_category`, several anomalies were discovered and addressed:
1. **Header Categories vs. Subcategories**:
   - Genuine root categories in AMS are identified by `control == "F"` and codes ending in `0000` (e.g., `100000 FURNITURE`, `260000 PENDINGIN RUANGAN`).
   - 24 rows in AMS had `parent` set to `NULL` but `control == "T"` (e.g., `AC Split`, `AC Portable`, `GENSET`, `CCTV`, `TV`). These are **subcategories** whose parent was omitted in the legacy system.
   - **Resolution**: Filter true categories using `control == "F"`. Relegate the 24 orphan subcategories to a dedicated staging backlog (`relegated_subcategories.csv`) so they can be properly imported into the `subcategories` table in `subcategory.ipynb`.
2. **Category Deduplication & Consolidation**:
   - `PROPERTI` (170000) and `GEDUNG` (220000) are merged into a single comprehensive category: **`Properti & Gedung`** (`PROP`).
   - `KENDARAAN` (200000) and `OTOMOTIF` (250000) are merged into: **`Kendaraan`** (`KEND`).
   - `PENDINGIN RUANGAN` (260000) is assigned standardized code **`PDGR`**.
   - `MESIN` (180000) is assigned standardized code **`MES`** (replacing `SIN`).
   - `ELEKTRONIK` (240000) is assigned standardized code **`ELEK`** (replacing `ELK`).
3. **Incomplete Categories Flagging**:
   - Categories with `0` attached subcategories (such as `210000 PURNAMA`) are marked with `subcat_count = 0` and `is_incomplete = True` in the staging output. This allows verifying whether any actual assets reference them before including them in production.
4. **Preserving Legacy References**:
   - To ensure assets and units can always join back to their category, the staging output tracks `ams_codes` (as an array/list) and exports a dedicated 1-to-1 relational lookup table (`category_ams_mapping.csv`).
5. **Output Organization**:
   - `data/smart/staging/category/`: Staging dataset, relational lookup mapping, and relegated subcategories backlog.
   - `data/smart/final/category/`: Production CSV (strictly the 5 target columns) and executable SQL script (`categories.sql`).


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

# ---------------------------------------------------------
# Dynamic Path Configurations
# Robustly resolves data directory regardless of whether
# current working directory is /home/jovyan, /home/jovyan/work,
# or the repository root.
# ---------------------------------------------------------
def get_data_dir():
    candidates = [
        Path("data"),                   # Running from project root or /home/jovyan
        Path("../data"),                # Running from notebooks/ or /home/jovyan/work
        Path("/home/jovyan/data"),      # Absolute path inside jupyter container
        Path("/home/ran/Devs/jupyter-ams-smart/data"), # Absolute path on host
    ]
    for p in candidates:
        if p.exists() and (p / "ams").exists():
            return p.resolve()
    return Path("../data").resolve()

DATA_DIR = get_data_dir()
RAW_DATA_PATH = DATA_DIR / "ams" / "tb_assets_category_202609291319.csv"

STAGING_DIR = DATA_DIR / "smart" / "staging" / "category"
FINAL_DIR = DATA_DIR / "smart" / "final" / "category"

# Ensure target directories exist
STAGING_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Paths initialized (CWD: {Path.cwd()}):")
print(f" - Data Root: {DATA_DIR}")
print(f" - Raw Data:  {RAW_DATA_PATH}")
print(f" - Staging:   {STAGING_DIR}")
print(f" - Final:     {FINAL_DIR}")


Paths initialized (CWD: /home/jovyan/work):
 - Data Root: /home/jovyan/data
 - Raw Data:  /home/jovyan/data/ams/tb_assets_category_202609291319.csv
 - Staging:   /home/jovyan/data/smart/staging/category
 - Final:     /home/jovyan/data/smart/final/category


In [2]:
# Load raw dataset, ensuring literal string 'NULL' and blanks are treated as NaN
df_raw = pd.read_csv(
    RAW_DATA_PATH,
    na_values=["NULL", "null", "NaN", "nan", ""],
    keep_default_na=True,
    dtype=str
)

# Strip whitespace from string columns
for col in df_raw.select_dtypes(include=["object"]).columns:
    df_raw[col] = df_raw[col].str.strip()

print(f"Loaded {len(df_raw)} total rows from AMS category dataset.")
display(df_raw.head(10))


Loaded 128 total rows from AMS category dataset.


,id,code,parent,pengelola,numbering_code,category,description,owner_create,create_date,control
0,51,100000,NaN,CFS,FUR,FURNITURE,FURITURE,302716,2018-04-05 19:22:48.000,F
1,52,110000,NaN,CFS,ACC,ACCESORIS,ACCESORIS,302716,2016-12-08 14:32:30.740,F
2,58,120000,NaN,ICT,COMP,COMPUTER,NaN,053573,2016-12-16 10:11:55.893,F
3,59,130000,NaN,ICT,MON,MONITOR,NaN,053573,2016-12-16 10:12:31.627,F
4,60,140000,NaN,ICT,PERP,PERIPHERAL,NaN,053573,2016-12-16 10:13:22.567,F
5,61,150000,NaN,ICT,PRNT,PRINTER,NaN,053573,2016-12-16 10:14:53.260,F
6,62,160000,NaN,ICT,SOFT,SOFTWARE,NaN,302716,2016-12-16 10:15:22.833,F
7,63,120003,COMP,ICT,DESK,DESKTOP,NaN,255578,2026-04-08 08:29:48.000,T
8,64,120004,COMP,ICT,NB,NOTEBOOK,NaN,255578,2026-04-08 08:30:07.000,T
9,65,130003,MON,ICT,CRT,TABUNG,NaN,255578,2026-04-08 08:32:21.000,T


## 3. Exploratory Analysis & Anomaly Detection

We examine the separation between header categories (`control == "F"`) and subcategories (`control == "T"`).
We also isolate the **24 orphan subcategories** that were mistakenly created with `parent = NULL` in AMS.


In [3]:
# Analyze control column breakdown
print("=== Control column value distribution ===")
print(df_raw["control"].value_counts(dropna=False))

# Identify root categories (control == 'F')
root_categories_raw = df_raw[df_raw["control"] == "F"].copy()
print(f"\nTrue header categories (control == 'F'): {len(root_categories_raw)} rows")

# Identify orphan subcategories (control == 'T' and parent is null)
orphan_subcats = df_raw[(df_raw["control"] == "T") & (df_raw["parent"].isna())].copy()
print(f"Orphan subcategories (control == 'T' and parent is NULL): {len(orphan_subcats)} rows\n")

# Map each orphan subcategory to its intended parent category based on code prefix
def determine_intended_parent(code_str):
    code_val = int(code_str)
    parent_code = (code_val // 10000) * 10000
    match = root_categories_raw[root_categories_raw["code"].astype(int) == parent_code]
    if not match.empty:
        return parent_code, match.iloc[0]["numbering_code"], match.iloc[0]["category"]
    return None, None, None

intended_parents = [determine_intended_parent(c) for c in orphan_subcats["code"]]
orphan_subcats["intended_parent_code"] = [p[0] for p in intended_parents]
orphan_subcats["intended_parent_num_code"] = [p[1] for p in intended_parents]
orphan_subcats["intended_parent_name"] = [p[2] for p in intended_parents]

# Fix spelling typos (e.g. AC FORTABLE -> AC Portable)
orphan_subcats["corrected_name"] = orphan_subcats["category"].replace({
    "AC FORTABLE": "AC Portable",
    "Celling Speaker": "Ceiling Speaker",
    "FURITURE": "Furniture"
})

# Add resolution reason for tracking
orphan_subcats["relegation_reason"] = (
    "Orphan subcategory in AMS (parent was NULL but control='T'). "
    "Relegated to subcategory table linked to parent code " + orphan_subcats["intended_parent_code"].astype(str)
)

print("Preview of Relegated Subcategories Backlog (first 10 rows):")
display(orphan_subcats[["id", "code", "numbering_code", "category", "corrected_name", "intended_parent_code", "intended_parent_name"]].head(10))


=== Control column value distribution ===
control
T    113
F     15
Name: count, dtype: int64

True header categories (control == 'F'): 15 rows
Orphan subcategories (control == 'T' and parent is NULL): 24 rows

Preview of Relegated Subcategories Backlog (first 10 rows):


,id,code,numbering_code,category,corrected_name,intended_parent_code,intended_parent_name
53,119,260001,ACS,AC Split,AC Split,260000,PENDINGIN RUANGAN
54,120,260002,AHU,AC AHU,AC AHU,260000,PENDINGIN RUANGAN
55,121,260003,AC,AC Standing,AC Standing,260000,PENDINGIN RUANGAN
63,1157,240012,FR,FINGERPRINT READER,FINGERPRINT READER,240000,ELEKTRONIK
68,1163,240001,CS,Celling Speaker,Ceiling Speaker,240000,ELEKTRONIK
70,1168,260005,ACF,AC FORTABLE,AC Portable,260000,PENDINGIN RUANGAN
71,1170,240018,MIC,MICROWAVE,MICROWAVE,240000,ELEKTRONIK
72,1171,180002,GEN,GENSET,GENSET,180000,MESIN
73,1172,180003,PAN,PANEL,PANEL,180000,MESIN
74,1173,240006,HD,HEAT DETECTOR,HEAT DETECTOR,240000,ELEKTRONIK


## 4. Master Category Configuration & Mapping
Here we define the target categories configuration.

### Merging & Code Standardization Rationale:
- **`PROP` (Properti & Gedung)**: Combines AMS `170000 PROPERTI` and `220000 GEDUNG`.
- **`KEND` (Kendaraan)**: Combines AMS `200000 KENDARAAN` and `250000 OTOMOTIF`.
- **`PDGR` (Pendingin Ruangan)**: Standardized 4-character code for `260000 PENDINGIN RUANGAN` (housing all AC subcategories).
- **`MES` (Mesin)**: Standardized 3-character code replacing legacy `SIN`.
- **`ELEK` (Elektronik)**: Standardized 4-character code replacing legacy `ELK`.
- **`PUR` (Purnama)**: Flagged as incomplete (`is_incomplete = True`) because it has 0 subcategories in AMS.


In [4]:
# Configuration dictionary for Master Categories
CATEGORY_CONFIG = [
    {
        "id": 1,
        "code": "FUR",
        "name": "Furniture",
        "ams_codes": [100000],
        "notes": "Perabotan kantor (kursi, meja, lemari, dll.)"
    },
    {
        "id": 2,
        "code": "ACC",
        "name": "Aksesoris",
        "ams_codes": [110000],
        "notes": "Aksesoris kantor, peralatan umum, dan fixture"
    },
    {
        "id": 3,
        "code": "COMP",
        "name": "Computer",
        "ams_codes": [120000],
        "notes": "Komputer desktop, notebook, server, workstation"
    },
    {
        "id": 4,
        "code": "MON",
        "name": "Monitor",
        "ams_codes": [130000],
        "notes": "Monitor tabung dan LCD"
    },
    {
        "id": 5,
        "code": "PERP",
        "name": "Peripheral",
        "ams_codes": [140000],
        "notes": "Perangkat pendukung IT, networking, multimedia, storage"
    },
    {
        "id": 6,
        "code": "PRNT",
        "name": "Printer",
        "ams_codes": [150000],
        "notes": "Printer inkjet, laserjet, plotter, ID card printer"
    },
    {
        "id": 7,
        "code": "SOFT",
        "name": "Software",
        "ams_codes": [160000],
        "notes": "Perangkat lunak engineering dan general"
    },
    {
        "id": 8,
        "code": "PROP",
        "name": "Properti & Gedung",
        "ams_codes": [170000, 220000],
        "notes": "Konsolidasi kategori properti dan gedung"
    },
    {
        "id": 9,
        "code": "MES",
        "name": "Mesin",
        "ams_codes": [180000],
        "notes": "Mesin dan utilitas gedung (genset, panel, pompa, lift)"
    },
    {
        "id": 10,
        "code": "KEND",
        "name": "Kendaraan",
        "ams_codes": [200000, 250000],
        "notes": "Konsolidasi kategori kendaraan operasional dan otomotif"
    },
    {
        "id": 11,
        "code": "ELEK",
        "name": "Elektronik",
        "ams_codes": [240000],
        "notes": "Peralatan elektronik umum, keamanan, dan multimedia (CCTV, TV)"
    },
    {
        "id": 12,
        "code": "PDGR",
        "name": "Pendingin Ruangan",
        "ams_codes": [260000],
        "notes": "Sistem tata udara dan pendingin ruangan (AC Split, AHU, dll.)"
    },
    {
        "id": 13,
        "code": "PUR",
        "name": "Purnama",
        "ams_codes": [210000],
        "notes": "Kategori legacy tanpa subkategori terdaftar; ditandai untuk review"
    }
]

print(f"Configured {len(CATEGORY_CONFIG)} master categories.")


Configured 13 master categories.


## 5. Data Transformation & Subcategory Activity Metrics

In this step, we:
1. Count the number of active subcategories associated with each category in AMS (both by `parent` code and by `code` prefix).
2. Mark categories with `subcat_count == 0` as `is_incomplete = True`.
3. Standardize timestamps to SQL Server format (`YYYY-MM-DD HH:MM:SS`).
4. Generate the **Staging DataFrame**, **1-to-1 Mapping DataFrame**, and **Final Target DataFrame**.


In [5]:
# Calculate subcategory counts per category in AMS
all_subcats = df_raw[df_raw["control"] == "T"].copy()

def get_subcat_count(ams_codes_list):
    count = 0
    for ac in ams_codes_list:
        prefix = str(ac)[:2]
        raw_match = df_raw[df_raw["code"].astype(int) == ac]
        num_code = raw_match.iloc[0]["numbering_code"] if not raw_match.empty else ""
        
        matched_subs = all_subcats[
            (all_subcats["parent"] == num_code) | 
            (all_subcats["code"].str.startswith(prefix))
        ]
        count += len(matched_subs.drop_duplicates(subset=["code"]))
    return count

def get_earliest_timestamp(ams_codes_list):
    dates = []
    for ac in ams_codes_list:
        match = df_raw[df_raw["code"].astype(int) == ac]
        if not match.empty and pd.notna(match.iloc[0]["create_date"]):
            dates.append(match.iloc[0]["create_date"])
    if dates:
        dt = min(pd.to_datetime(dates))
        return dt.strftime("%Y-%m-%d %H:%M:%S")
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def get_ams_ids(ams_codes_list):
    ids = []
    for ac in ams_codes_list:
        match = df_raw[df_raw["code"].astype(int) == ac]
        if not match.empty:
            ids.append(int(match.iloc[0]["id"]))
    return ids

# Build Staging & Mapping rows
staging_rows = []
mapping_rows = []

for item in CATEGORY_CONFIG:
    cat_id = item["id"]
    code = item["code"]
    name = item["name"]
    ams_codes = item["ams_codes"]
    ams_ids = get_ams_ids(ams_codes)
    
    subcat_count = get_subcat_count(ams_codes)
    is_incomplete = bool(subcat_count == 0)
    created_at = get_earliest_timestamp(ams_codes)
    updated_at = created_at
    
    staging_rows.append({
        "id": cat_id,
        "code": code,
        "name": name,
        "ams_codes": str(ams_codes),
        "ams_ids": str(ams_ids),
        "subcat_count": subcat_count,
        "is_incomplete": is_incomplete,
        "created_at": created_at,
        "updated_at": updated_at,
        "notes": item["notes"]
    })
    
    for ac, ai in zip(ams_codes, ams_ids):
        mapping_rows.append({
            "ams_code": ac,
            "ams_id": ai,
            "category_id": cat_id,
            "new_code": code,
            "new_name": name
        })

df_staging = pd.DataFrame(staging_rows)
df_mapping = pd.DataFrame(mapping_rows)

# Build Final Production DataFrame (Strictly 5 Target Columns)
# Incomplete categories (subcat_count == 0) can be excluded from production:
EXCLUDE_INCOMPLETE_FROM_FINAL = True

if EXCLUDE_INCOMPLETE_FROM_FINAL:
    df_final = df_staging[~df_staging["is_incomplete"]][["id", "code", "name", "created_at", "updated_at"]].copy()
else:
    df_final = df_staging[["id", "code", "name", "created_at", "updated_at"]].copy()

# Ensure sequential IDs starting at 1
df_final["id"] = range(1, len(df_final) + 1)

print("=== Transformed Staging Categories ===")
display(df_staging[["id", "code", "name", "ams_codes", "subcat_count", "is_incomplete"]])

print("\n=== Final Production Categories (Target MSSQL Schema) ===")
display(df_final)


=== Transformed Staging Categories ===


,id,code,name,ams_codes,subcat_count,is_incomplete
0,1,FUR,Furniture,[100000],8,False
1,2,ACC,Aksesoris,[110000],43,False
2,3,COMP,Computer,[120000],4,False
3,4,MON,Monitor,[130000],2,False
4,5,PERP,Peripheral,[140000],15,False
5,6,PRNT,Printer,[150000],6,False
6,7,SOFT,Software,[160000],2,False
7,8,PROP,Properti & Gedung,"[170000, 220000]",1,False
8,9,MES,Mesin,[180000],4,False
9,10,KEND,Kendaraan,"[200000, 250000]",2,False



=== Final Production Categories (Target MSSQL Schema) ===


,id,code,name,created_at,updated_at
0,1,FUR,Furniture,2018-04-05 19:22:48,2018-04-05 19:22:48
1,2,ACC,Aksesoris,2016-12-08 14:32:30,2016-12-08 14:32:30
2,3,COMP,Computer,2016-12-16 10:11:55,2016-12-16 10:11:55
3,4,MON,Monitor,2016-12-16 10:12:31,2016-12-16 10:12:31
4,5,PERP,Peripheral,2016-12-16 10:13:22,2016-12-16 10:13:22
5,6,PRNT,Printer,2016-12-16 10:14:53,2016-12-16 10:14:53
6,7,SOFT,Software,2016-12-16 10:15:22,2016-12-16 10:15:22
7,8,PROP,Properti & Gedung,2022-07-01 13:28:04,2022-07-01 13:28:04
8,9,MES,Mesin,2017-01-05 17:29:43,2017-01-05 17:29:43
9,10,KEND,Kendaraan,2023-06-13 14:02:32,2023-06-13 14:02:32


## 6. Data Validation & Quality Checks
We run assertions and integrity checks before exporting to ensure:
1. `code` values are strictly unique and $\le 4$ characters (`VARCHAR(4)`).
2. No null values exist in required fields (`id`, `code`, `name`, `created_at`, `updated_at`).
3. Output matches SQL Server target schema.


In [6]:
# 1. Check code length constraint (VARCHAR(4))
max_code_len = df_final["code"].str.len().max()
assert max_code_len <= 4, f"Error: Maximum code length exceeds 4 characters ({max_code_len})"

# 2. Check code uniqueness
duplicate_codes = df_final[df_final["code"].duplicated(keep=False)]
assert duplicate_codes.empty, f"Error: Duplicate category codes detected: {duplicate_codes['code'].tolist()}"

# 3. Check for NULL values in required columns
null_counts = df_final.isna().sum()
assert null_counts.sum() == 0, f"Error: Found NULL values in final dataset: {null_counts.to_dict()}"

# 4. Check sequential IDs
assert df_final["id"].is_monotonic_increasing, "Error: IDs are not monotonic increasing"

print("ALL DATA QUALITY CHECKS PASSED SUCCESSFULLY!")
print(f" - Total Production Categories: {len(df_final)}")
print(f" - Max Code Length: {max_code_len} characters")
print(f" - Unique Codes: {df_final['code'].nunique()} / {len(df_final)}")
print(f" - Incomplete categories flagged in staging: {df_staging['is_incomplete'].sum()} ({df_staging[df_staging['is_incomplete']]['name'].tolist()})")


ALL DATA QUALITY CHECKS PASSED SUCCESSFULLY!
 - Total Production Categories: 12
 - Max Code Length: 4 characters
 - Unique Codes: 12 / 12
 - Incomplete categories flagged in staging: 1 (['Purnama'])


## 7. Export Outputs

We export staging files and final production files in **two separate cells** as requested:
1. **Cell 8**: Staging files (`data/smart/staging/category/`)
   - `categories.csv`
   - `category_ams_mapping.csv`
   - `relegated_subcategories.csv`
2. **Cell 9**: Final production files (`data/smart/final/category/`)
   - `categories.csv`
   - `categories.sql` (Executable MS SQL Server script with `IDENTITY_INSERT`)


In [7]:
# ---------------------------------------------------------
# Export Staging Outputs to data/smart/staging/category/
# ---------------------------------------------------------
staging_cat_path = STAGING_DIR / "categories.csv"
staging_map_path = STAGING_DIR / "category_ams_mapping.csv"
staging_relegated_path = STAGING_DIR / "relegated_subcategories.csv"

# 1. Staging categories with legacy codes & activity metrics
df_staging.to_csv(staging_cat_path, index=False)

# 2. 1-to-1 Relational lookup mapping table
df_mapping.to_csv(staging_map_path, index=False)

# 3. Relegated subcategories backlog for subcategory.ipynb
orphan_subcats.to_csv(staging_relegated_path, index=False)

print(f"Staging files successfully exported to {STAGING_DIR.resolve()}:")
print(f" 1. {staging_cat_path.name} ({len(df_staging)} rows)")
print(f" 2. {staging_map_path.name} ({len(df_mapping)} rows)")
print(f" 3. {staging_relegated_path.name} ({len(orphan_subcats)} rows)")


Staging files successfully exported to /home/jovyan/data/smart/staging/category:
 1. categories.csv (13 rows)
 2. category_ams_mapping.csv (15 rows)
 3. relegated_subcategories.csv (24 rows)


In [8]:
# ---------------------------------------------------------
# Export Production Outputs to data/smart/final/category/
# ---------------------------------------------------------
final_csv_path = FINAL_DIR / "categories.csv"
final_sql_path = FINAL_DIR / "categories.sql"

# 1. Target CSV for SQL Server Import (Strict 5 Columns)
df_final.to_csv(final_csv_path, index=False)

# 2. Generate SQL Server INSERT Script
sql_statements = [
    "-- ========================================================",
    "-- Master Categories Data Migration Script for MS SQL Server",
    f"-- Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    "-- Target Table: categories",
    "-- ========================================================",
    "SET IDENTITY_INSERT [categories] ON;",
    "GO",
    ""
]

for _, row in df_final.iterrows():
    c_id = row["id"]
    c_code = str(row["code"]).replace("'", "''")
    c_name = str(row["name"]).replace("'", "''")
    c_created = row["created_at"]
    c_updated = row["updated_at"]
    sql = f"INSERT INTO [categories] ([id], [code], [name], [created_at], [updated_at]) VALUES ({c_id}, '{c_code}', N'{c_name}', '{c_created}', '{c_updated}');"
    sql_statements.append(sql)

sql_statements.extend([
    "",
    "GO",
    "SET IDENTITY_INSERT [categories] OFF;",
    "GO"
])

with open(final_sql_path, "w", encoding="utf-8") as f:
    f.write("\n".join(sql_statements))

print(f"Production files successfully exported to {FINAL_DIR.resolve()}:")
print(f" 1. {final_csv_path.name} ({len(df_final)} rows)")
print(f" 2. {final_sql_path.name} ({len(df_final)} SQL insert statements)")
print("\n--- SQL Script Preview ---")
print("\n".join(sql_statements[:15]))


Production files successfully exported to /home/jovyan/data/smart/final/category:
 1. categories.csv (12 rows)
 2. categories.sql (12 SQL insert statements)

--- SQL Script Preview ---
-- ========================================================
-- Master Categories Data Migration Script for MS SQL Server
-- Generated: 2026-09-29 08:05:08
-- Target Table: categories
-- ========================================================
SET IDENTITY_INSERT [categories] ON;
GO

INSERT INTO [categories] ([id], [code], [name], [created_at], [updated_at]) VALUES (1, 'FUR', N'Furniture', '2018-04-05 19:22:48', '2018-04-05 19:22:48');
INSERT INTO [categories] ([id], [code], [name], [created_at], [updated_at]) VALUES (2, 'ACC', N'Aksesoris', '2016-12-08 14:32:30', '2016-12-08 14:32:30');
INSERT INTO [categories] ([id], [code], [name], [created_at], [updated_at]) VALUES (3, 'COMP', N'Computer', '2016-12-16 10:11:55', '2016-12-16 10:11:55');
INSERT INTO [categories] ([id], [code], [name], [created_at], [upd